# W6D4 — Linear Probing: What Is a Representation Worth? — Lab

**Week 6 · Day 4 · Representation Learning** · Lab

Four representations, one weak classifier, one table. That table is why this week reuses week 4's
400 images and refuses to substitute anything more convenient.

Today you put your Tuesday autoencoder codes, your Wednesday MAE features, DINOv2's frozen
features and a supervised ResNet-18's features through the **same linear probe on the same folds
W4D5 used** — folds rebuilt from W4D5's own grouping code and checked against a recorded hash,
because a comparison across two weeks on two different splits is not a comparison.

Then the skill that outlives the week: three different evaluations of the *same* features, an
example of an evaluation that looks like a probe and is not, and the label-efficiency curve that
is the entire economic argument for self-supervision in one picture.

Assignment A6 is issued today.

**Time budget:** ~115 minutes. Sections 1–2 are the lab; Section 3 is a stretch you may finish at home.

<div dir="rtl" align="right">

# الأسبوع ٦ · اليوم ٤ — الفحص الخطيّ: كم يساوي التمثيل؟

**الأسبوع السادس · اليوم الرابع · تعلّم التمثيل** · معمل

أربعة تمثيلات، ومصنّف ضعيف واحد، وجدول واحد. وذلك الجدول هو سبب إعادة هذا الأسبوع استخدام صور
الأسبوع الرابع الأربعمئة ورفضه استبدال أي شيء أيسر بها.

اليوم تُمرّر شيفرات مُرمِّزك الذاتي من الثلاثاء، وتمثيلات مُرمِّزك المُقنَّع من الأربعاء، وتمثيلات DINOv2
المجمّدة، وتمثيلات ResNet-18 المُشرَف، عبر **الفحص الخطيّ نفسه على التقسيمات نفسها التي استعملها
الأسبوع الرابع اليوم الخامس** — تقسيمات مُعاد بناؤها من كود التجميع الخاص به ومفحوصة مقابل بصمة
مُسجَّلة، لأن مقارنةً بين أسبوعين على تقسيمين مختلفين ليست مقارنة.

ثم المهارة التي تبقى بعد الأسبوع: ثلاثة تقييمات مختلفة للتمثيلات **نفسها**، ومثالٌ على تقييم يبدو
فحصًا وليس به، ومنحنى كفاءة التسميات الذي هو كل الحجّة الاقتصادية للإشراف الذاتي في صورة واحدة.

ويُسلَّم التكليف السادس اليوم.

**الزمن المتوقّع:** نحو ١١٥ دقيقة. القسمان الأول والثاني هما المعمل، والقسم الثالث إضافي يمكن إكماله في المنزل.

</div>

> **This is your lab notebook.** Work through the hints — they tell you what to do and where
> to look, not what to type. Stuck for more than ten minutes on one task? Open the `_guided`
> version. That is not cheating; sitting stuck in silence is the only mistake. The full
> solution is released at the end of the day.

<div dir="rtl" align="right">

> **هذا دفتر المعمل الخاص بك.** اعمل وفق الإرشادات — فهي تخبرك بما يجب فعله وأين تبحث، لا بما
> تكتبه حرفيًا. إذا توقّفت أكثر من عشر دقائق عند مهمة واحدة فافتح نسخة `_guided`؛ هذا ليس غشًّا،
> والخطأ الوحيد هو أن تبقى متوقّفًا بصمت. ويُنشر الحل الكامل في نهاية اليوم.

</div>

## Learning objectives

By the end of this lab you can:

- Compute the cross-entropy between a teacher and a student distribution, and recognise both
  collapse modes from their loss values.
- Say why the sharp-collapse loss being *lower* than the honest one is the problem centring exists
  to solve.
- Extract frozen features from a pretrained backbone and **prove** they are frozen rather than
  assert it.
- Rebuild another lab's train/validation split exactly, and check it with a hash before comparing
  anything.
- Fit a linear probe and report a representation's value as one number on stated folds.
- Say what a k-NN score, a probe score and a fine-tune score each measure, and where they disagree.
- Recognise an evaluation that is not a probe, and say why its number must not be reported as one.
- Read a label-efficiency curve and turn it into a sentence about a budget.

<div dir="rtl" align="right">

## أهداف التعلّم

في نهاية هذا المعمل تستطيع:

- أن تحسب الانتروبيا المتقاطعة بين توزيع المعلّم وتوزيع الطالب، وأن تتعرّف على نمطَي الانهيار من
  قيمتَي خسارتهما.
- أن تقول لماذا كون خسارة الانهيار الحادّ **أدنى** من الخسارة الأمينة هو المشكلة التي وُجد التمركز
  لحلّها.
- أن تستخرج تمثيلات مجمّدة من نموذج مُدرَّب مسبقًا وأن **تُبرهن** تجميدها لا أن تدّعيه.
- أن تُعيد بناء تقسيم معملٍ آخر بالضبط، وأن تفحصه ببصمة قبل أن تقارن شيئًا.
- أن تُلائم فحصًا خطيًا وتُبلّغ عن قيمة تمثيلٍ في رقم واحد على تقسيمات مذكورة.
- أن تقول ماذا يقيس كلٌّ من درجة الجيران الأقرب ودرجة الفحص ودرجة الضبط الدقيق، وأين تختلف.
- أن تتعرّف على تقييم ليس فحصًا، وأن تقول لماذا لا يجوز ذكر رقمه على أنه فحص.
- أن تقرأ منحنى كفاءة التسميات وتحوّله إلى جملة عن ميزانية.

</div>

## About the data

`small_image_5class`, for the fourth day, and this is the day the repetition pays.

**The folds.** W4D5 did not split these 400 images naively. It grouped them by perceptual hash
first — because twelve of the eighty `pizza` images are near-duplicates, and a duplicate pair
straddling the split inflates every accuracy on the page — and then split the *groups*, stratified,
at seed 42. This lab rebuilds that split with the same code and checks the result against
`w4d5_folds.json`, a recorded artefact holding the hash of the two index arrays and W4D5's four
accuracies. **If the hash does not match, the comparison is void and the sanity check says so.**

**Loaded from earlier labs**, each with the standard `solutions_cache` fallback if you missed a day:

- `ae_codes.parquet` — Tuesday's 32-dim autoencoder codes.
- `mae_features.parquet` — Wednesday's 768-dim MAE `[CLS]` features.
- `w4d5_folds.json` and `transfer_curves.parquet` — W4D5's split and its numbers.
- `unseen_predictions.parquet` — what W4D5's fine-tuned model did on the 25 unseen images.

**One honest caveat about Tuesday's codes.** That autoencoder was trained on a plain stratified
split, not on today's grouped one, so a handful of today's validation images were in its unlabelled
training set. No label was involved, so this is not label leakage — but it does mean the
autoencoder row is, if anything, flattered relative to the others. It is the weakest row by a long
way regardless, which is why the finding survives the caveat.

`unseen_images` — the 25 held-out images, three of which are elephants that no lab has ever trained
on.

**First run downloads** `facebook/dinov2-small` — 88 MB. Extracting features for all 400 images
takes about **8 seconds**; the light fine-tune in task 2.4 is the longest cell at roughly **30
seconds**; everything else is a logistic regression on 400 rows.

<div dir="rtl" align="right">

## عن البيانات

`small_image_5class` لليوم الرابع، وهذا هو اليوم الذي يُثمر فيه التكرار.

**التقسيمات.** لم يقسم الأسبوع الرابع اليوم الخامس هذه الصور الأربعمئة تقسيمًا ساذجًا. بل جمّعها
بالتجزيء الإدراكي أولًا — لأن اثنتي عشرة من صور `pizza` الثمانين شبه مكرّرة، وزوجٌ مكرّر يتوزّع بين
شطرَي التقسيم يرفع كل دقّة في الصفحة زورًا — ثم قسم **المجموعات** تقسيمًا طبقيًا عند البذرة ٤٢. ويُعيد
هذا المعمل بناء ذلك التقسيم بالكود نفسه ويفحص الناتج مقابل `w4d5_folds.json`، وهو أثر مُسجَّل يحمل
بصمة مصفوفتَي الفهارس ودقّات الأسبوع الرابع الأربع. **وإن لم تُطابق البصمة بطلت المقارنة، ويقول فحص
السلامة ذلك.**

**ويُحمَّل من معامل سابقة**، ولكلٍّ رجوعه المعتاد إلى `solutions_cache` إن تغيّبتَ يومًا:

- `ae_codes.parquet` — شيفرات الثلاثاء ذات الـ٣٢ بُعدًا.
- `mae_features.parquet` — تمثيلات الأربعاء `[CLS]` ذات الـ٧٦٨ بُعدًا.
- `w4d5_folds.json` و`transfer_curves.parquet` — تقسيم الأسبوع الرابع وأرقامه.
- `unseen_predictions.parquet` — ما فعله نموذج الأسبوع الرابع المضبوط على الصور الخمس والعشرين.

**وتحفّظ صادق واحد على شيفرات الثلاثاء.** دُرِّب ذلك المُرمِّز على تقسيم طبقيّ عادي لا على تقسيم اليوم
المُجمَّع، فكانت حفنة من صور تحقّق اليوم في مجموعة تدريبه غير المُسمّاة. ولم تدخل تسمية، فليس هذا
تسريب تسميات — لكنه يعني أن صفّ المُرمِّز الذاتي مُجامَل إن كان ثمّة مجاملة. وهو أضعف الصفوف بفارق كبير
على أي حال، ولهذا تصمد النتيجة أمام التحفّظ.

`unseen_images` — الصور الخمس والعشرون المحجوزة، ثلاثٌ منها أفيال لم يتدرّب عليها أي معمل قط.

**التشغيل الأول ينزّل** النموذج `facebook/dinov2-small` — ‏٨٨ ميجابايت. واستخراج التمثيلات للصور
الأربعمئة نحو **ثماني ثوانٍ**؛ والضبط الخفيف في المهمة ٢٫٤ أطول خلية بنحو **ثلاثين ثانية**؛ وكل ما
عدا ذلك انحدار لوجستي على أربعمئة صف.

</div>

## Setup

Everything a probe needs is in `scikit-learn`. The only torch in this lab is the backbone, and it
spends most of the afternoon in `eval()` with its gradients off.

<div dir="rtl" align="right">

## الإعداد

كل ما يحتاجه الفحص في `scikit-learn`. وtorch الوحيد في هذا المعمل هو النموذج الأساس، ويقضي معظم
الظهيرة في وضع التقييم وتدرّجاته مُطفأة.

</div>

In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
from aiep.data import get_dataset_dir, load_artefact
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, check_close, check_shape, report
from aiep.viz import use_course_style, savefig

ensure("transformers", "torch", "torchvision", "scikit-learn", "matplotlib", "pandas", "pyarrow")
seed_everything(42)

import hashlib
import json
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

use_course_style()
np.set_printoptions(precision=4, suppress=True)

SEED = 42
DINO_CHECKPOINT = "facebook/dinov2-small"
FEATURE_BATCH = 25

IMAGE_ROOT = get_dataset_dir("small_image_5class") / "images"
PATHS = sorted(IMAGE_ROOT.rglob("*.jpg"))
LABELS = np.array([p.parent.name for p in PATHS])
CLASSES = sorted(set(LABELS))
CHANCE = 1.0 / len(CLASSES)

# W4D5's recorded split and its numbers — today's comparison is against these.
W4D5 = json.loads(load_artefact("w4d5_folds.json").read_text(encoding="utf-8"))
W4D5_CURVES = pd.read_parquet(load_artefact("transfer_curves.parquet"))
W4D5_BEST = W4D5_CURVES.groupby("run").val_accuracy.max()

print(f"{len(PATHS)} images, {len(CLASSES)} classes {CLASSES}, chance {CHANCE:.2f}")
print(f"W4D5 recorded {W4D5['n_train']} train / {W4D5['n_val']} val, "
      f"folds hash {W4D5['folds_hash']}")
print(W4D5_BEST.to_string())
print(versions(), "| device:", device())

## Section 1 — Warm-up: collapse, and why a probe is weak  (≈25 min)

Two parts, both straight from the morning, and both are three lines of arithmetic.

**Part A — the collapse arithmetic.** DINO trains a student to match a teacher's distribution over
prototypes. The teacher says `[0.7, 0.2, 0.1]`, the student says `[0.4, 0.4, 0.2]`, and the
cross-entropy `−Σ pₜ log pₛ` is **0.9856**. That is an honest disagreement between two networks
that are still learning.

Now the two ways it can cheat. If both collapse onto the *same sharp* output `[0.98, 0.01, 0.01]`,
the loss is **0.1119** — far lower than the honest one, and the model has stopped saying anything
about the image. If both collapse onto the *uniform* output, the loss is **1.0986**, which is
exactly `ln 3` and is *worse* than being honest.

The asymmetry is the whole point. Uniform collapse punishes itself and needs no defence. Sharp
collapse is **rewarded** by the objective, so it needs a mechanism — centring — to stop it. Print
all three and check the direction.

**Part B — why the probe is deliberately weak.** Four two-dimensional points, twice. Arranged with
the classes on opposite sides, a logistic regression scores **1.00**. Arranged as XOR, the same
classifier on the same four points scores **0.50**, which is chance. Nothing changed but where the
features put the points — and a stronger classifier would have found the XOR structure itself and
reported 1.00 both times, telling you nothing about the representation. The probe's weakness *is*
the measurement.

<div dir="rtl" align="right">

## القسم الأول — الإحماء: الانهيار، ولماذا الفحص ضعيف (نحو ٢٥ دقيقة)

جزآن، كلاهما من الصباح مباشرةً، وكلاهما ثلاثة أسطر من الحساب.

**الجزء أ — حساب الانهيار.** يُدرّب DINO طالبًا ليُطابق توزيع معلّمٍ على نماذج أوّلية. يقول المعلّم
`[0.7, 0.2, 0.1]`، ويقول الطالب `[0.4, 0.4, 0.2]`، والانتروبيا المتقاطعة `−Σ pₜ log pₛ` تساوي
**٠٫٩٨٥٦**. وهذا خلاف أمين بين شبكتين ما زالتا تتعلّمان.

والآن طريقتا الغشّ. إن انهار الاثنان إلى الخرج **الحادّ نفسه** `[0.98, 0.01, 0.01]` كانت الخسارة
**٠٫١١١٩** — أدنى بكثير من الأمينة، وقد كفّ النموذج عن قول أي شيء عن الصورة. وإن انهارا إلى الخرج
**المنتظم** كانت الخسارة **١٫٠٩٨٦**، وهي `ln 3` بالضبط، وهي **أسوأ** من الأمانة.

وعدم التناظر هو المقصد كله. فالانهيار المنتظم يعاقب نفسه ولا يحتاج دفاعًا. والانهيار الحادّ
**يُكافئه** الهدف، فيحتاج آليةً — التمركز — لإيقافه. اطبع الثلاثة وافحص الاتجاه.

**الجزء ب — لماذا الفحص ضعيف عن قصد.** أربع نقاط ثنائية البُعد، مرّتين. فبترتيبٍ يضع الفئتين على
جانبين متقابلين يسجّل الانحدار اللوجستي **١٫٠٠**. وبترتيب XOR يسجّل المصنّف نفسه على النقاط الأربع
نفسها **٠٫٥٠**، وهي المصادفة. ولم يتغيّر إلا موضع النقاط في التمثيل — ولوجد مصنّفٌ أقوى بنية XOR
بنفسه وأبلغ ١٫٠٠ في الحالتين، فلا يقول لك شيئًا عن التمثيل. فضعف الفحص **هو** القياس.

</div>

In [ ]:
TEACHER = np.array([0.7, 0.2, 0.1])
STUDENT = np.array([0.4, 0.4, 0.2])
SHARP = np.array([0.98, 0.01, 0.01])
UNIFORM = np.array([1 / 3, 1 / 3, 1 / 3])


def cross_entropy(teacher, student):
    """-sum(p_t * log p_s), the three-term version from the slide."""
    return float(-(teacher * np.log(student)).sum())


HONEST_LOSS = cross_entropy(TEACHER, STUDENT)
SHARP_LOSS = cross_entropy(SHARP, SHARP)
UNIFORM_LOSS = cross_entropy(UNIFORM, UNIFORM)

print(f"honest disagreement   : {HONEST_LOSS:.4f}")
print(f"sharp collapse        : {SHARP_LOSS:.4f}   <- LOWER, and the model said nothing")
print(f"uniform collapse      : {UNIFORM_LOSS:.4f}   = ln 3 = {np.log(3):.4f}")

SHARP_IS_REWARDED = SHARP_LOSS < HONEST_LOSS
UNIFORM_IS_PUNISHED = UNIFORM_LOSS > HONEST_LOSS
print(f"\nsharp collapse scores better than honesty : {SHARP_IS_REWARDED}  <- centring exists "
      f"for this")
print(f"uniform collapse scores worse than honesty: {UNIFORM_IS_PUNISHED}  <- punishes itself")

In [ ]:
# Four two-dimensional "frozen features", two arrangements, one classifier.
POINTS = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
SEPARABLE_Y = np.array([0, 0, 1, 1])          # split by the first coordinate
XOR_Y = np.array([0, 1, 1, 0])                # opposite corners share a class

SEPARABLE_SCORE = LogisticRegression().fit(POINTS, SEPARABLE_Y).score(POINTS, SEPARABLE_Y)
XOR_SCORE = LogisticRegression().fit(POINTS, XOR_Y).score(POINTS, XOR_Y)

fig, axes = plt.subplots(1, 2, figsize=(8.5, 4))
for ax, y, name, score in [(axes[0], SEPARABLE_Y, "separable", SEPARABLE_SCORE),
                           (axes[1], XOR_Y, "XOR", XOR_SCORE)]:
    for label, marker in [(0, "o"), (1, "s")]:
        chosen = POINTS[y == label]
        ax.scatter(chosen[:, 0], chosen[:, 1], marker=marker, s=180, label=f"class {label}")
    ax.set_title(f"{name} — probe accuracy {score:.2f}")
    ax.set_xlim(-0.4, 1.4)
    ax.set_ylim(-0.4, 1.4)
    ax.legend(fontsize=8)
fig.suptitle("same probe, same four points, same labels — different arrangement")
fig.tight_layout()
savefig(fig, "probe_intuition.png")
plt.show()

print(f"separable arrangement: {SEPARABLE_SCORE:.2f}")
print(f"XOR arrangement      : {XOR_SCORE:.2f}   (chance on a balanced binary problem)")
print("\nA three-layer head would report 1.00 on both, and tell you nothing.")

## Section 2 — Core: six tasks  (≈60 min)

1. Frozen DINOv2 features, with the freeze **proved**.
2. W4D5's folds, rebuilt and hash-checked, and the first probe number.
3. The comparison table — four representations, one probe, identical folds.
4. Three evaluations of the same features: k-NN, probe, light fine-tune.
5. The invalid probe.
6. The 25 unseen images, and what a confidence distribution should look like.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ست مهام (نحو ٦٠ دقيقة)

١. تمثيلات DINOv2 مجمّدة، مع **برهان** التجميد.
٢. تقسيمات الأسبعوع الرابع، مُعاد بناؤها ومفحوصة البصمة، وأول رقم فحص.
٣. جدول المقارنة — أربعة تمثيلات، وفحص واحد، وتقسيمات متطابقة.
٤. ثلاثة تقييمات للتمثيلات نفسها: الجيران الأقرب، والفحص، وضبط دقيق خفيف.
٥. الفحص الباطل.
٦. الصور الخمس والعشرون غير المرئية، وكيف ينبغي أن يبدو توزيع الثقة.

</div>

### Task 2.1 — frozen features, proved frozen

Load DINOv2 and extract one 384-dim vector per image.

Then prove the backbone is frozen, in two ways that catch two different mistakes:

1. **Every parameter has `requires_grad == False`.** A backbone with live gradients that happens
   not to be optimised today is still not a frozen backbone: attach an optimiser to
   `model.parameters()` by habit and you have silently fine-tuned it.
2. **The model is in `eval()` mode.** This is the subtler one. In `train()` mode dropout is active
   and batch-norm statistics update, so the "frozen" features you extract depend on the batch they
   were extracted in — the same image gets a different vector depending on what was next to it.
   The features are then not a function of the image at all, and nothing downstream is reproducible.

Both are one-line assertions and both belong in every probing script you ever write. This is the
guard, not the ceremony.

<div dir="rtl" align="right">

### المهمة ٢٫١ — تمثيلات مجمّدة، مع برهان التجميد

حمّل DINOv2 واستخرج متّجهًا واحدًا بـ٣٨٤ بُعدًا لكل صورة.

ثم برهِن أن النموذج الأساس مجمّد، بطريقتين تُمسكان خطأين مختلفين:

١. **كل معامل عنده `requires_grad == False`.** فنموذج أساس بتدرّجات حيّة صادف ألّا يُحسَّن اليوم ليس
   نموذجًا مجمّدًا: أوصِل مُحسِّنًا بـ`model.parameters()` بحكم العادة تكن قد ضبطته دقيقًا صامتًا.
٢. **النموذج في وضع `eval()`.** وهذه أدقّ. ففي وضع `train()` يعمل الإسقاط (Dropout) وتتحدّث إحصاءات
   التوحيد الدفعي، فتعتمد التمثيلات «المجمّدة» التي تستخرجها على الدفعة التي استُخرجت فيها — فتنال
   الصورة نفسها متّجهًا مختلفًا بحسب ما جاورها. فلا تكون التمثيلات دالّةً للصورة أصلًا، ولا يكون أي
   شيء لاحق قابلًا لإعادة الإنتاج.

وكلاهما فحصٌ في سطر، وكلاهما ينتمي إلى كل نصّ فحصٍ تكتبه. وهذا حَرَسٌ لا مراسم.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) AutoModel and AutoImageProcessor from transformers, loaded from DINO_CHECKPOINT.
# 2) Set requires_grad = False on every parameter, call .eval(), and check both with a
#    generator expression before you extract anything.
# 3) Extract in batches of FEATURE_BATCH under torch.no_grad(); pooler_output is the
#    [CLS]-derived image vector and is what a probe is normally fitted on.
# Search: "dinov2 transformers frozen feature extraction pooler_output"
# https://huggingface.co/docs/transformers/model_doc/dinov2
#
# ١) `AutoModel` و`AutoImageProcessor` من `transformers`، محمّلين من `DINO_CHECKPOINT`.
# ٢) اضبط `requires_grad = False` على كل معامل، ونادِ `.eval()`، وافحص الأمرين بتعبير
#    مولّد قبل أن تستخرج شيئًا.
# ٣) استخرج على دفعات بحجم `FEATURE_BATCH` تحت `torch.no_grad()`؛ و`pooler_output` هو
#    متّجه الصورة المشتقّ من `[CLS]` وهو ما يُلائَم عليه الفحص عادةً.
# ابحث عن: "dinov2 transformers frozen feature extraction pooler_output"
# https://huggingface.co/docs/transformers/model_doc/dinov2
# ────────────────────────────────────────────────────────────────────

# TODO: Load DINOv2, freeze every parameter, put it in eval mode, assert both facts, and extract one feature vector per image in batches.
# مهمة: حمّل DINOv2، وجمّد كل معامل، وضعه في وضع التقييم، وافحص الأمرين، واستخرج متّجه تمثيل واحدًا لكل صورة على دفعات.

### Task 2.2 — W4D5's folds, rebuilt and checked

This is the cell the whole week rests on, and it contains no machine learning at all.

Rebuild W4D5's split with W4D5's own procedure: perceptual-hash every image, join any two images
within a Hamming distance of 8 into a group, then split the **groups** — stratified, seed 42.
Grouping first is what stops a near-duplicate pair from landing on both sides of the split.

Then hash the two index arrays and compare against the hash recorded in `w4d5_folds.json`. If it
matches, every number in today's table is directly comparable with W4D5's. If it does not, you have
a different split and the table means nothing — so this is an assertion, not a printout.

Fit the first probe on DINOv2's features on those folds. One number, and it should land above
W4D5's fine-tune.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — تقسيمات الأسبوع الرابع، مُعاد بناؤها ومفحوصة

هذه هي الخلية التي يقوم عليها الأسبوع كله، وليس فيها تعلّم آلة إطلاقًا.

أعِد بناء تقسيم الأسبوع الرابع بإجرائه نفسه: جزّئ كل صورة تجزيئًا إدراكيًا، واضمم أي صورتين تفصل
بينهما مسافة هامينغ ثمانية فأقلّ في مجموعة واحدة، ثم قسّم **المجموعات** تقسيمًا طبقيًا عند البذرة ٤٢.
والتجميع أولًا هو ما يمنع زوجًا شبه مكرّر من الوقوع على جانبَي التقسيم.

ثم جزّئ مصفوفتَي الفهارس وقارِن بالبصمة المُسجَّلة في `w4d5_folds.json`. فإن طابقت كان كل رقم في جدول
اليوم قابلًا للمقارنة المباشرة بأرقام الأسبوع الرابع. وإن لم تُطابق فعندك تقسيم آخر ولا يعني الجدول
شيئًا — فهذا فحصٌ لا طباعة.

ولائم أول فحص على تمثيلات DINOv2 على تلك التقسيمات. رقم واحد، وينبغي أن يقع فوق ضبط الأسبوع الرابع
الدقيق.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) The dHash is the one from W4D4: greyscale, resize to 9x8, compare each pixel with its
#    right-hand neighbour, flatten to 64 bits.
# 2) Join groups by walking the pairs whose Hamming distance is <= 8, then
#    train_test_split the unique group ids with test_size=0.2, stratify and random_state.
# 3) Hash "train,indices|val,indices" with hashlib.sha256 and take the first 16 hex
#    characters — the recorded value in W4D5["folds_hash"] was made the same way.
# Search: "perceptual hash group split prevent duplicate leakage"
# https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
#
# ١) التجزيء الإدراكي هو نفسه من الأسبوع الرابع اليوم الرابع: تدرّج رمادي، وتحجيم إلى ٩×٨،
#    ومقارنة كل بكسل بجاره الأيمن، وطيّ إلى ٦٤ بتًّا.
# ٢) اضمم المجموعات بالمرور على الأزواج التي مسافتها ٨ فأقلّ، ثم `train_test_split` على
#    معرّفات المجموعات الفريدة بـ`test_size=0.2` مع التطبيق والبذرة.
# ٣) جزّئ النصّ `"train,indices|val,indices"` بـ`hashlib.sha256` وخذ أول ١٦ خانة ستّ عشرية
#    — فالقيمة المُسجَّلة في `W4D5["folds_hash"]` صُنعت بالطريقة نفسها.
# ابحث عن: "perceptual hash group split prevent duplicate leakage"
# https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
# ────────────────────────────────────────────────────────────────────

# TODO: Rebuild W4D5's grouped split, hash the two index arrays, compare against the recorded hash, then fit a linear probe on DINOv2's features and report its accuracy.
# مهمة: أعِد بناء تقسيم الأسبوع الرابع المُجمَّع، وجزّئ مصفوفتَي الفهارس، وقارن بالبصمة المُسجَّلة، ثم لائم فحصًا خطيًا على تمثيلات DINOv2 وأبلغ عن دقّته.
def linear_probe(features, train_index=None, val_index=None):
    """Validation accuracy of one linear classifier on frozen features."""
    train_index = TRAIN_INDEX if train_index is None else train_index
    val_index = VAL_INDEX if val_index is None else val_index
    pipeline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
    pipeline.fit(features[train_index], LABELS[train_index])
    return pipeline, float(pipeline.score(features[val_index], LABELS[val_index]))

### Task 2.3 — the comparison table

Five rows, one probe, identical folds. **This is the moment of the week; if time is short, cut a
later task rather than a row here.**

| representation | where it came from |
|---|---|
| autoencoder codes, 32-dim | Tuesday, trained by you, no labels |
| MAE features, 768-dim | Wednesday, pretrained on ImageNet, no labels |
| DINOv2 features, 384-dim | pretrained self-supervised, no labels |
| ResNet-18 penultimate, 512-dim | pretrained **supervised** on ImageNet labels |
| W4D5 fine-tuned | reference row: not a probe, a whole trained model |

Extract the ResNet-18 features yourself — they are the same frozen ImageNet features W4D5's run B
put a head on, so the probe number should land near W4D5's 0.925 without being identical: a
logistic regression fitted to convergence is a stronger head than eight epochs of SGD on an
augmented stream. Say that in one line rather than pretending the two should match.

Record the trainable parameter count next to each accuracy. That column is half the argument: the
probe's head is a few thousand numbers and the fine-tune's is eleven million.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — جدول المقارنة

خمسة صفوف، وفحص واحد، وتقسيمات متطابقة. **وهذه لحظة الأسبوع؛ فإن ضاق الوقت فاحذف مهمّة لاحقة لا صفًّا
من هنا.**

| التمثيل | من أين جاء |
|---|---|
| شيفرات المُرمِّز الذاتي، ٣٢ بُعدًا | الثلاثاء، درّبتَه أنت، بلا تسميات |
| تمثيلات المُرمِّز المُقنَّع، ٧٦٨ بُعدًا | الأربعاء، مُدرَّب مسبقًا على ImageNet، بلا تسميات |
| تمثيلات DINOv2، ٣٨٤ بُعدًا | مُدرَّب مسبقًا بإشراف ذاتي، بلا تسميات |
| ResNet-18 ما قبل الأخيرة، ٥١٢ بُعدًا | مُدرَّب مسبقًا **بإشراف** على تسميات ImageNet |
| الأسبوع الرابع المضبوط دقيقًا | صفّ مرجعي: ليس فحصًا بل نموذجًا مُدرَّبًا كاملًا |

استخرج تمثيلات ResNet-18 بنفسك — فهي تمثيلات ImageNet المجمّدة نفسها التي وضع عليها الأسبوع الرابع
رأسًا في تشغيلته «ب»، فينبغي أن يقع رقم الفحص قرب ‎٠٫٩٢٥‎ دون أن يطابقه: فانحدار لوجستي مُلائَم إلى
التقارب رأسٌ أقوى من ثماني دورات نزول تدرّجي على تدفّق مُعزَّز. قل ذلك في سطر بدل التظاهر بأنهما
يجب أن يتطابقا.

وسجّل عدد المعاملات القابلة للتدريب بجوار كل دقّة. فذلك العمود نصف الحجّة: رأس الفحص بضعة آلاف عدد،
ورأس الضبط الدقيق أحد عشر مليونًا.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) load_artefact("ae_codes.parquet") and load_artefact("mae_features.parquet") — take
#    the code_ and f columns with DataFrame.filter and check the row order matches PATHS.
# 2) ResNet-18: torchvision IMAGENET1K_V1 weights, fc replaced with nn.Identity, the
#    weights' own .transforms() for preprocessing, eval mode, no_grad.
# 3) One loop over the four representations calling linear_probe, and one extra row read
#    straight out of W4D5_BEST for the fine-tune reference.
# Search: "torchvision resnet18 penultimate features nn.Identity"
# https://docs.pytorch.org/vision/stable/models.html
#
# ١) `load_artefact("ae_codes.parquet")` و`load_artefact("mae_features.parquet")` — خذ
#    أعمدة `code_` و`f` بـ`DataFrame.filter` وتأكّد أن ترتيب الصفوف يطابق `PATHS`.
# ٢) ResNet-18: أوزان `IMAGENET1K_V1` من torchvision، و`fc` مستبدلة بـ`nn.Identity`،
#    ومعالجة `.transforms()` الخاصة بالأوزان، ووضع التقييم، و`no_grad`.
# ٣) حلقة واحدة على التمثيلات الأربعة تنادي `linear_probe`، وصفٌّ إضافي يُقرأ من
#    `W4D5_BEST` مرجعًا للضبط الدقيق.
# ابحث عن: "torchvision resnet18 penultimate features nn.Identity"
# https://docs.pytorch.org/vision/stable/models.html
# ────────────────────────────────────────────────────────────────────

# TODO: Load Tuesday's and Wednesday's features, extract ResNet-18's, probe all four on the same folds, and build one table with W4D5's fine-tune as a fifth reference row.
# مهمة: حمّل تمثيلات الثلاثاء والأربعاء، واستخرج تمثيلات ResNet-18، وافحص الأربعة على التقسيمات نفسها، وابنِ جدولًا واحدًا مع ضبط الأسبوع الرابع صفًّا مرجعيًا خامسًا.

### Task 2.4 — three evaluations, three questions

Same DINOv2 features, three ways to score them.

- **k-NN, k=5.** No training at all. It asks: *are same-class images already near each other?* It
  measures the metric structure of the space directly and cannot compensate for anything.
- **Linear probe.** One matrix. It asks: *is class information linearly readable?* — a weaker
  requirement than metric structure, because a linear map can rescale and rotate first.
- **Light fine-tune of the last block.** 1.8 million parameters move. It asks: *how good is this
  checkpoint as a starting point?* It is not a measure of the representation, because the
  representation changes during the measurement.

Report all three. Expect them to be close on DINOv2, and note the parameter counts — the fine-tune
moves roughly a thousand times more numbers to get there, and on 320 images it does not necessarily
get there at all. Write one sentence per method saying what it measures.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — ثلاثة تقييمات، ثلاثة أسئلة

تمثيلات DINOv2 نفسها، وثلاث طرق لتقييمها.

- **الجيران الأقرب، k=5.** بلا تدريب إطلاقًا. ويسأل: *هل صور الفئة الواحدة متجاورة أصلًا؟* فيقيس
  بنية المسافة في الفضاء مباشرةً ولا يستطيع تعويض شيء.
- **الفحص الخطيّ.** مصفوفة واحدة. ويسأل: *هل معلومة الفئة مقروءة خطيًا؟* — وهو شرط أضعف من بنية
  المسافة، لأن التحويل الخطيّ يستطيع إعادة القياس والدوران أولًا.
- **ضبط دقيق خفيف للكتلة الأخيرة.** ‏١٫٨ مليون معامل تتحرّك. ويسأل: *كم هذا النموذج المحفوظ جيّدًا
  نقطةَ انطلاق؟* وليس قياسًا للتمثيل، لأن التمثيل يتغيّر أثناء القياس.

أبلغ عن الثلاثة. وتوقّع تقاربها على DINOv2، ولاحظ أعداد المعاملات — فالضبط الدقيق يُحرّك نحو ألف ضعف
من الأعداد ليصل، وعلى ثلاثمئة وعشرين صورة قد لا يصل أصلًا. اكتب جملة لكل طريقة تقول ماذا تقيس.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) KNeighborsClassifier(n_neighbors=5) inside the same StandardScaler pipeline, fitted
#    on the training rows and scored on the validation rows.
# 2) For the fine-tune, wrap the backbone in a small nn.Module with a Linear head, unfreeze
#    only backbone.encoder.layer[-1], and use AdamW at 1e-4 for three epochs.
# 3) Remember torch.set_grad_enabled — the setup cell did not turn gradients off globally
#    today, but the extraction ran under no_grad and the fine-tune must not.
# Search: "pytorch freeze all but last transformer block fine-tune"
# https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html
#
# ١) `KNeighborsClassifier(n_neighbors=5)` داخل مسار `StandardScaler` نفسه، مُلائَمًا على
#    صفوف التدريب ومُقيَّمًا على صفوف التحقّق.
# ٢) وللضبط الدقيق، لُفّ النموذج الأساس في وحدة صغيرة برأس `Linear`، وأطلِق
#    `backbone.encoder.layer[-1]` وحدها، واستعمل `AdamW` عند ‎1e-4‎ لثلاث دورات.
# ٣) وتذكّر `torch.set_grad_enabled` — فخلية الإعداد لم تُطفئ التدرّجات عالميًا اليوم، لكن
#    الاستخراج جرى تحت `no_grad` ويجب ألّا يجري الضبط الدقيق كذلك.
# ابحث عن: "pytorch freeze all but last transformer block fine-tune"
# https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html
# ────────────────────────────────────────────────────────────────────

# TODO: Score the DINOv2 features with k-NN (k=5) and with the probe, then fine-tune only the backbone's last block plus a linear head and report all three with their parameter costs.
# مهمة: قيّم تمثيلات DINOv2 بالجيران الأقرب (k=5) وبالفحص، ثم اضبط الكتلة الأخيرة وحدها مع رأس خطّي، وأبلغ عن الثلاثة مع كلفة معاملات كلٍّ.

### Task 2.5 — the invalid probe

Now do the thing you must never report.

Put a three-layer MLP head on **Tuesday's autoencoder codes** — the weakest representation in the
table — and record its training accuracy and its validation accuracy next to the linear probe's.

Look at the training column first. The linear probe cannot even fit the training set; the MLP fits
it perfectly. That is the tell. The head has enough capacity to memorise 320 points, so the number
it produces is a statement about the head, not about the 32 numbers it was given.

On this data the validation accuracy also *falls*, which is the tidiest possible ending — but note
that it did not have to. On a bigger dataset the MLP head would have gone up, and it would still
have been an invalid probe result, for exactly the same reason. **The reason a deep head disqualifies
a probe number is not that it overfits. It is that it can untangle a representation the probe was
supposed to be measuring** — and then you learn that the head is capable, which you already knew.

Write your two sentences in the markdown cell below. The row goes into the artefact with
`is_probe = False`, and that flag is asserted at the bottom of the notebook.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — الفحص الباطل

افعل الآن ما لا يجوز أن تُبلّغ عنه أبدًا.

ضع رأسًا بثلاث طبقات على **شيفرات مُرمِّزك الذاتي من الثلاثاء** — أضعف تمثيل في الجدول — وسجّل دقّته
على التدريب ودقّته على التحقّق بجوار دقّتَي الفحص الخطيّ.

وانظر إلى عمود التدريب أولًا. فالفحص الخطيّ لا يستطيع حتى ملاءمة مجموعة التدريب؛ والرأس متعدّد الطبقات
يلائمها ملاءمةً تامّة. وهذه هي العلامة. فسعة الرأس تكفي لحفظ ثلاثمئة وعشرين نقطة، فيكون الرقم الذي
يُنتجه قولًا عن الرأس لا عن الأعداد الاثنين والثلاثين التي أُعطيها.

وعلى هذه البيانات تهبط دقّة التحقّق أيضًا، وهي أنظف نهاية ممكنة — لكن لاحظ أن ذلك لم يكن لازمًا. فعلى
بيانات أكبر كان الرأس ليرتفع، ولظلّ نتيجة فحصٍ باطلة، للسبب نفسه بالضبط. **فسبب إبطال الرأس العميق
لرقم الفحص ليس أنه يُفرِط في الملاءمة. بل أنه يستطيع فكّ تشابك تمثيلٍ كان الفحص يقيسه** — فتتعلّم
عندئذ أن الرأس قادر، وهو ما كنت تعرفه.

اكتب جملتيك في خلية Markdown أدناه. ويدخل الصف في الأثر بـ`is_probe = False`، ويُفحص ذلك الوسم في
أسفل الدفتر.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) MLPClassifier with two or three hidden layers inside the same StandardScaler
#    pipeline, max_iter high enough to converge, random_state=SEED.
# 2) Score it on the training rows as well as the validation rows — the training column is
#    where the diagnosis is, not the validation one.
# 3) Do the same for the linear probe on the same features so the two are side by side,
#    then append a row to TABLE with is_probe set to False.
# Search: "sklearn MLPClassifier train accuracy overfitting small dataset"
# https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html
#
# ١) `MLPClassifier` بطبقتين أو ثلاث مخفيّة داخل مسار `StandardScaler` نفسه، و`max_iter`
#    كافٍ للتقارب، و`random_state=SEED`.
# ٢) قيّمه على صفوف التدريب كما على صفوف التحقّق — فعمود التدريب هو موضع التشخيص لا عمود
#    التحقّق.
# ٣) افعل الشيء نفسه للفحص الخطيّ على التمثيلات نفسها ليكونا جنبًا إلى جنب، ثم أضف صفًّا إلى
#    `TABLE` بـ`is_probe` مضبوطًا على `False`.
# ابحث عن: "sklearn MLPClassifier train accuracy overfitting small dataset"
# https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html
# ────────────────────────────────────────────────────────────────────

# TODO: Fit a 3-layer MLP head and a linear probe on the autoencoder codes, report train and validation accuracy for both, and append the MLP row flagged as not a probe.
# مهمة: لائم رأسًا بثلاث طبقات وفحصًا خطيًا على شيفرات المُرمِّز الذاتي، وأبلغ عن دقّتَي التدريب والتحقّق لكليهما، وأضف صفّ الرأس موسومًا بأنه ليس فحصًا.

**Your two sentences.** Why must the MLP-head number not be reported as a probe result? Answer in
terms of what is being measured, not in terms of overfitting.

<div dir="rtl" align="right">

**جملتاك.** لماذا لا يجوز ذكر رقم الرأس متعدّد الطبقات على أنه نتيجة فحص؟ أجب من جهة ما يُقاس، لا من
جهة الإفراط في الملاءمة.

</div>

### Task 2.6 — the 25 unseen images

The honest final check, and the same one W4D5 ran.

Predict on `unseen_images` with the DINOv2 probe. Twenty-two are from the five trained classes;
**three are elephants**, a class nothing this week has ever seen. `labels.csv` marks them
`in_or_out = out`.

Plot the confidence distribution for the two groups and compare against `unseen_predictions.parquet`
— W4D5's fine-tuned model gave the in-class images a mean confidence of about 0.92 and the
elephants about 0.54. A five-class softmax always returns five numbers that sum to one, so the
question is never "did it say elephant" — it cannot — but "did it at least look uncertain".

Report both means and say which model separates the two groups better. Neither is allowed to be
right about the elephants. One of them is allowed to be visibly unsure.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — الصور الخمس والعشرون غير المرئية

التحقّق النهائي الصادق، وهو نفسه الذي أجراه الأسبوع الرابع اليوم الخامس.

تنبّأ على `unseen_images` بفحص DINOv2. اثنتان وعشرون من الفئات الخمس المُدرَّبة؛ و**ثلاث أفيال**، وهي
فئة لم يرَها شيء هذا الأسبوع. ويسمها `labels.csv` بـ`in_or_out = out`.

ارسم توزيع الثقة للمجموعتين وقارنه بـ`unseen_predictions.parquet` — فنموذج الأسبوع الرابع المضبوط
أعطى صور الفئات ثقةً متوسّطها نحو ‎٠٫٩٢‎ والأفيال نحو ‎٠٫٥٤‎. ودالّة softmax بخمس فئات تُعيد دائمًا خمسة
أعداد تجمع إلى واحد، فالسؤال ليس «هل قال فيل» — فلا يستطيع — بل «هل بدا غير واثق على الأقلّ».

أبلغ عن المتوسّطين وقل أيّ النموذجين يفصل المجموعتين فصلًا أفضل. ولا يُسمح لأيّهما أن يكون مصيبًا في
الأفيال. ويُسمح لأحدهما أن يكون غير واثق بوضوح.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) get_dataset_dir("unseen_images") holds images/ and labels.csv; read the csv and keep
#    the file order it gives you so the in/out column lines up.
# 2) Extract DINOv2 features for those 25 the same way, then use the fitted probe's
#    predict_proba and take the max per row as the confidence.
# 3) Load W4D5's unseen_predictions.parquet with load_artefact and put the two confidence
#    distributions on one figure, split by in_or_out.
# Search: "sklearn predict_proba max confidence out of distribution"
# https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
#
# ١) يحوي `get_dataset_dir("unseen_images")` مجلّد `images/` وملف `labels.csv`؛ اقرأ الملف
#    وأبقِ ترتيب الملفات الذي يعطيكه ليتحاذى عمود الداخل والخارج.
# ٢) استخرج تمثيلات DINOv2 لهذه الخمس والعشرين بالطريقة نفسها، ثم استعمل `predict_proba`
#    من الفحص المُلائَم وخذ الأكبر في كل صف ثقةً.
# ٣) حمّل `unseen_predictions.parquet` من الأسبوع الرابع بـ`load_artefact` وضع توزيعَي
#    الثقة في شكل واحد مفصولين بـ`in_or_out`.
# ابحث عن: "sklearn predict_proba max confidence out of distribution"
# https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
# ────────────────────────────────────────────────────────────────────

# TODO: Predict on the 25 unseen images with the DINOv2 probe, compare the in-class and out-of-class confidence distributions, and put W4D5's numbers beside yours.
# مهمة: تنبّأ على الصور الخمس والعشرين بفحص DINOv2، وقارن توزيعَي الثقة للفئات وخارجها، وضع أرقام الأسبوع الرابع بجوار أرقامك.

## Section 3 — Stretch: the label-efficiency curve  (≈30 min)

This is the week's economic argument, and it is one plot.

Fit the probe on 10, 25, 50, 100 and all 320 training images, for **DINOv2's features** and for
**your autoencoder codes**, and plot validation accuracy against label count. Use the same
randomly drawn subsets for both so the comparison is paired.

Expect the DINOv2 curve to be far above the autoencoder's at every point and close to flat by 50
labels — most of the accuracy is available from a number of labels a person could produce over
lunch, because the representation did the work that labels usually have to do. Expect it to be
bumpy: with a 10-label fit and an 80-image validation set, one image is 1.25 accuracy points, and
saying so is part of reading the curve honestly.

Then the two sentences that curve justifies. Frame them as a decision: you have an image problem, a
fixed budget, and an annotator. What do you do first, and what would have to be true for the answer
to change?

**Capstone link.** If your capstone has images and few labels, your first model is frozen features
plus a probe — not a fine-tune. This table is the argument, and it takes an afternoon to reproduce
on your own data. A6 is issued today and asks for exactly that.

<div dir="rtl" align="right">

## القسم الثالث — التوسّع: منحنى كفاءة التسميات (نحو ٣٠ دقيقة)

هذه حجّة الأسبوع الاقتصادية، وهي رسمٌ واحد.

لائم الفحص على ١٠ و٢٥ و٥٠ و١٠٠ ثم كل الصور الثلاثمئة والعشرين، على **تمثيلات DINOv2** وعلى
**شيفرات مُرمِّزك الذاتي**، وارسم دقّة التحقّق مقابل عدد التسميات. واستعمل المجموعات الجزئية المسحوبة
نفسها للاثنين لتكون المقارنة مزدوجة.

وتوقّع أن يكون منحنى DINOv2 فوق منحنى المُرمِّز بفارق كبير عند كل نقطة، وشبه مستوٍ عند خمسين تسمية —
فمعظم الدقّة متاح من عدد تسميات يستطيع شخص إنتاجه في استراحة الغداء، لأن التمثيل أدّى العمل الذي
تؤدّيه التسميات عادةً. وتوقّعه متعرّجًا: فبملاءمةٍ على عشر تسميات ومجموعة تحقّقٍ من ثمانين صورة تساوي
الصورة الواحدة ١٫٢٥ نقطة دقّة، وقول ذلك جزءٌ من قراءة المنحنى قراءةً صادقة.

ثم الجملتان اللتان يُبرّرهما ذلك المنحنى. صُغهما قرارًا: عندك مسألة صور، وميزانية ثابتة، ومُسمٍّ. ماذا
تفعل أولًا، وما الذي يجب أن يصحّ ليتغيّر الجواب؟

**صلة مشروع التخرّج.** إن كان في مشروعك صور وتسميات قليلة فنموذجك الأول تمثيلات مجمّدة وفحص — لا ضبط
دقيق. وهذا الجدول هو الحجّة، وإعادة إنتاجه على بياناتك تستغرق ظهيرة. ويُسلَّم التكليف السادس اليوم
ويطلب ذلك بعينه.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Draw the subsets once, outside the representation loop, so both curves are fitted on
#    exactly the same images at each budget.
# 2) At 10 labels a stratified draw is not guaranteed to contain all five classes — draw
#    with a seeded generator and print how many classes each budget actually got.
# 3) Plot both curves on one axis with the label count on a log scale and a dotted line at
#    chance.
# Search: "label efficiency curve frozen features few labels"
# https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.choice.html
#
# ١) اسحب المجموعات الجزئية مرّةً خارج حلقة التمثيلات، ليُلائَم المنحنيان على الصور نفسها
#    تمامًا عند كل ميزانية.
# ٢) عند عشر تسميات لا يضمن السحب وجود الفئات الخمس كلها — فاسحب بمولّد مُبذَّر واطبع كم فئة
#    نالتها كل ميزانية فعلًا.
# ٣) ارسم المنحنيين على محور واحد بعدد التسميات على مقياس لوغاريتمي وخطٍّ منقّط عند
#    المصادفة.
# ابحث عن: "label efficiency curve frozen features few labels"
# https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.choice.html
# ────────────────────────────────────────────────────────────────────

# TODO: Fit the probe at each label budget on the same drawn subsets for DINOv2 and for the autoencoder codes, and plot both curves against the label count.
# مهمة: لائم الفحص عند كل ميزانية تسميات على المجموعات الجزئية المسحوبة نفسها لـDINOv2 ولشيفرات المُرمِّز الذاتي، وارسم المنحنيين مقابل عدد التسميات.

**Your two sentences.** You have an image problem, a fixed budget and one annotator. What do you do
first, and what would have to be true for that answer to change?

<div dir="rtl" align="right">

**جملتاك.** عندك مسألة صور، وميزانية ثابتة، ومُسمٍّ واحد. ماذا تفعل أولًا، وما الذي يجب أن يصحّ ليتغيّر
ذلك الجواب؟

</div>

## Save your artefact

`probe_results.parquet` — every row of today's table: the representation, the evaluation type, the
folds hash, the accuracy, the trainable parameter count, and `is_probe`. **The folds hash is in
every row on purpose.** A number without its folds is not a result, and six months from now the
hash is what tells you whether two tables can be put side by side.

`dino_features.parquet` — the 384-dim frozen features for all 400 images. **Tomorrow loads this
file** to compare CLIP's zero-shot accuracy against today's probe on identical images.

<div dir="rtl" align="right">

## احفظ أثرك

`probe_results.parquet` — كل صف من جدول اليوم: التمثيل، ونوع التقييم، وبصمة التقسيمات، والدقّة، وعدد
المعاملات القابلة للتدريب، و`is_probe`. **وبصمة التقسيمات في كل صف عن قصد.** فالرقم بلا تقسيماته ليس
نتيجة، وبعد ستّة أشهر تكون البصمة هي ما يخبرك هل يجوز وضع جدولين جنبًا إلى جنب.

`dino_features.parquet` — التمثيلات المجمّدة ذات الـ٣٨٤ بُعدًا لكل الصور الأربعمئة. **ويُحمّل الغد هذا
الملف** ليقارن دقّة CLIP دون تدريب بفحص اليوم على الصور نفسها.

</div>

In [ ]:
RESULTS = pd.concat([
    TABLE,
    EVALUATIONS.assign(representation="DINOv2 features", dim=DINO_FEATURES.shape[1],
                       folds_hash=FOLDS_HASH, is_probe=EVALUATIONS.evaluation == "linear probe",
                       labels_in_pretraining="self-supervised, pretrained")
    [["representation", "evaluation", "dim", "folds_hash", "accuracy", "trainable_params",
      "labels_in_pretraining", "is_probe"]],
], ignore_index=True).drop_duplicates(subset=["representation", "evaluation"], keep="first")

RESULTS_PATH = ARTEFACT_DIR / "probe_results.parquet"
RESULTS.to_parquet(RESULTS_PATH, index=False)
EFFICIENCY.to_parquet(ARTEFACT_DIR / "label_efficiency.parquet", index=False)

features_frame = pd.concat([
    pd.DataFrame({"file": FILES, "label": LABELS,
                  "split": np.where(np.isin(np.arange(len(PATHS)), TRAIN_INDEX), "train", "val")}),
    pd.DataFrame(DINO_FEATURES, columns=[f"d{i:03d}" for i in range(DINO_FEATURES.shape[1])]),
], axis=1)
FEATURES_PATH = ARTEFACT_DIR / "dino_features.parquet"
features_frame.to_parquet(FEATURES_PATH, index=False)

print(RESULTS[["representation", "evaluation", "accuracy", "is_probe"]].to_string(index=False))
print(f"\n{RESULTS_PATH.name}: {len(RESULTS)} rows, all stamped with folds hash {FOLDS_HASH}")
print(f"{FEATURES_PATH.name}: {features_frame.shape[0]} x {DINO_FEATURES.shape[1]} features")

## Sanity check

<div dir="rtl" align="right">

## فحص سلامة

</div>

In [ ]:
check(np.isclose(HONEST_LOSS, 0.9856, atol=5e-4) and np.isclose(SHARP_LOSS, 0.1119, atol=5e-4)
      and np.isclose(UNIFORM_LOSS, 1.0986, atol=5e-4),
      f"the warm-up must reproduce the slide's three losses 0.9856, 0.1119 and 1.0986 — got "
      f"{HONEST_LOSS:.4f}, {SHARP_LOSS:.4f}, {UNIFORM_LOSS:.4f}",
      f"يجب أن يُعيد الإحماء خسائر الشريحة الثلاث ‎٠٫٩٨٥٦‎ و‎٠٫١١١٩‎ و‎١٫٠٩٨٦‎ — والناتج "
      f"{HONEST_LOSS:.4f} و{SHARP_LOSS:.4f} و{UNIFORM_LOSS:.4f}")

check(SHARP_IS_REWARDED and UNIFORM_IS_PUNISHED,
      f"sharp collapse must score LOWER than honest disagreement ({SHARP_LOSS:.4f} < "
      f"{HONEST_LOSS:.4f}) and uniform collapse HIGHER ({UNIFORM_LOSS:.4f} > {HONEST_LOSS:.4f}). "
      f"The direction is the lesson: the objective rewards one of the two failures, which is why "
      f"centring exists",
      f"يجب أن تكون خسارة الانهيار الحادّ أدنى من الخلاف الأمين ({SHARP_LOSS:.4f} < "
      f"{HONEST_LOSS:.4f}) وخسارة الانهيار المنتظم أعلى ({UNIFORM_LOSS:.4f} > {HONEST_LOSS:.4f}). "
      f"والاتجاه هو الدرس: فالهدف يكافئ أحد الإخفاقين، ولهذا وُجد التمركز")

check(np.isclose(SEPARABLE_SCORE, 1.00) and np.isclose(XOR_SCORE, 0.50),
      f"the probe intuition must give 1.00 on the separable arrangement and 0.50 on XOR — got "
      f"{SEPARABLE_SCORE:.2f} and {XOR_SCORE:.2f}",
      f"يجب أن يُعطي مثال الفحص ‎١٫٠٠‎ في الترتيب القابل للفصل و‎٠٫٥٠‎ في XOR — والناتج "
      f"{SEPARABLE_SCORE:.2f} و{XOR_SCORE:.2f}")

check(ALL_FROZEN and IN_EVAL_MODE and TRAINABLE_IN_BACKBONE == 0,
      f"every backbone parameter must have requires_grad False and the model must be in eval mode "
      f"— frozen: {ALL_FROZEN}, eval: {IN_EVAL_MODE}, trainable: {TRAINABLE_IN_BACKBONE:,}. A probe "
      f"with a live backbone is not a probe",
      f"يجب أن يكون `requires_grad` في كل معامل من النموذج الأساس `False` وأن يكون النموذج في وضع "
      f"التقييم — مجمّد: {ALL_FROZEN}، تقييم: {IN_EVAL_MODE}، قابل للتدريب: "
      f"{TRAINABLE_IN_BACKBONE:,}. والفحص بنموذج أساس حيّ ليس فحصًا")

check(FOLDS_MATCH and ROWS_ALIGN,
      f"the rebuilt folds must hash to W4D5's recorded {W4D5['folds_hash']} — got {FOLDS_HASH} — "
      f"and the loaded feature tables must be in the same image order (aligned: {ROWS_ALIGN}). If "
      f"either fails, today's table is not comparable with W4D5's and the week's whole comparison "
      f"is void",
      f"يجب أن تُجزّئ التقسيمات المُعاد بناؤها إلى بصمة الأسبوع الرابع المُسجَّلة {W4D5['folds_hash']} — "
      f"والناتج {FOLDS_HASH} — وأن تكون جداول التمثيلات المُحمَّلة بترتيب الصور نفسه (متحاذية: "
      f"{ROWS_ALIGN}). فإن أخفق أحدهما لم يكن جدول اليوم قابلًا للمقارنة وبطلت مقارنة الأسبوع كلها")

check(len(RESULTS[RESULTS.is_probe]) >= 4
      and set(RESULTS[RESULTS.is_probe].folds_hash) == {FOLDS_HASH},
      f"all four representations must appear as probe rows on one identical folds hash — got "
      f"{len(RESULTS[RESULTS.is_probe])} probe rows on hashes "
      f"{sorted(set(RESULTS[RESULTS.is_probe].folds_hash))}",
      f"يجب أن تظهر التمثيلات الأربعة صفوف فحصٍ على بصمة تقسيماتٍ واحدة — والناتج "
      f"{len(RESULTS[RESULTS.is_probe])} صف فحص على البصمات "
      f"{sorted(set(RESULTS[RESULTS.is_probe].folds_hash))}")

check(DINO_PROBE > float(W4D5_BEST.max()) and DINO_PROBE > CHANCE,
      f"a linear probe on frozen self-supervised features must beat W4D5's best supervised run on "
      f"the same folds — got {DINO_PROBE:.4f} against {W4D5_BEST.max():.4f}. That single "
      f"comparison is why this week reuses week 4's images",
      f"يجب أن يتجاوز الفحص الخطيّ على تمثيلات ذاتية الإشراف مجمّدة أفضل تشغيلة مُشرَفة في الأسبوع "
      f"الرابع على التقسيمات نفسها — والناتج {DINO_PROBE:.4f} مقابل {W4D5_BEST.max():.4f}. وهذه "
      f"المقارنة وحدها سبب إعادة هذا الأسبوع استخدام صور الأسبوع الرابع")

check(MLP_MEMORISES and not RESULTS.loc[RESULTS.evaluation.str.contains("NOT A PROBE"),
                                        "is_probe"].any(),
      f"the MLP head must fit the training set better than the linear one ({MLP_TRAIN:.3f} against "
      f"{LINEAR_TRAIN:.3f}) and its row must be flagged is_probe = False in the artefact",
      f"يجب أن يلائم الرأس متعدّد الطبقات مجموعة التدريب أفضل من الخطّي ({MLP_TRAIN:.3f} مقابل "
      f"{LINEAR_TRAIN:.3f}) وأن يكون صفّه موسومًا `is_probe = False` في الأثر")

report()

## What's next

**W6D5 — CLIP, and search across two kinds of data.** Tomorrow a model embeds images and *text* into
one shared space, which means you can search a picture collection by typing a sentence. You build
the retrieval by hand: two hundred pairs, one similarity matrix, read one way for text-to-image and
the other way for image-to-text.

It also closes today's table. CLIP classifies these same five classes with **zero** training images
— just the words "a photo of a bus" — and the number lands below today's probe. Below is the point:
you will put the three numbers in one row, spend zero labels for one of them, and then find where
CLIP breaks, which is the graded task and the honest end of the week.

`dino_features.parquet` is on disk. Bring it.

<div dir="rtl" align="right">

## ما التالي

**الأسبوع ٦ اليوم ٥ — CLIP، والبحث عبر نوعين من البيانات.** غدًا يُضمّن نموذجٌ الصورَ **والنصّ** في
فضاء مشترك واحد، ومعنى ذلك أنك تستطيع البحث في مجموعة صور بكتابة جملة. وتبني الاسترجاع بيدك: مئتا
زوج، ومصفوفة تشابه واحدة، تُقرأ في اتجاه للبحث من النص إلى الصورة وفي الآخر من الصورة إلى النص.

وهو يُغلق جدول اليوم أيضًا. فـCLIP يُصنّف هذه الفئات الخمس نفسها بـ**صفر** صورة تدريب — بكلمات
«صورة لحافلة» فحسب — ويقع الرقم دون فحص اليوم. والدونية هي المقصد: ستضع الأرقام الثلاثة في صفّ واحد،
وتُنفق صفر تسمية على أحدها، ثم تجد أين ينكسر CLIP، وهي المهمّة المُقيَّمة ونهاية الأسبوع الصادقة.

وملف `dino_features.parquet` على القرص. أحضِره.

</div>